In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import calitp_data_analysis.magics

In [2]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
from siuba import *

fs = get_fs()

In [3]:
import altair as alt
import folium

In [4]:
import _replica_utils

In [5]:
from IPython.display import HTML

In [6]:
from calitp_data_analysis import calitp_color_palette as cp

In [7]:
pd.set_option("display.max_columns", None)

In [8]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"


In [9]:
display(HTML("<h1>Origin-Destination Big Data Analysis: Los Angeles POIs</h1>"))

In [10]:
display(HTML("This analysis uses Replica Data to identify the top trip start points within a Regional Transportation Planning Agency (RTPA) asnd determine what types of trips are occuring when and where."))

In [11]:
shape_data_name = "origins/replica-stm_origin_la_980035-03_04_26-origin_layer.zip"

origins_name = "replica-stm_origin_la_980035-03_04_26-trips_dataset.zip"

In [12]:
origins = _replica_utils.read_in_and_prep_replica_data_w_shp(origins_name, shape_data_name, file_type="")

In [13]:
# len(origins)

In [14]:
display(HTML("<h2><strong>Trip Counts</strong></h2>"))

In [15]:
display(HTML(f"The number of trips Traveling <strong>From</strong> the top 20 locations is <strong>{len(origins)}</strong>"))


In [16]:
with get_fs().open(f"{gcs_path}{shape_data_name}") as f:
        blkgr = to_snakecase(gpd.read_file(f))

In [17]:
blkgr.sample()

,geoname,customgeoid,centroidlat,centroidlon,squaremiles,tripsbyorigin,tripsbyorigin_sqmi,geometry
18,8729a1d64ffffff,608718350687666200,34.0191,-118.2924,2.2257,14293,6421.8898,"POLYGON ((-118.28409 34.00774, -118.27623 34.0..."


In [18]:
origins.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,geometry
234855,18145709824179637482,"1 (Tract 9800.28, Los Angeles, CA)","9800.28 (Los Angeles, CA)","Los Angeles County, CA",California,"4 (Tract 423.11, Orange, CA)","423.11 (Orange, CA)","Orange County, CA",California,private_auto,home,region_arrival,16:56:00,18:17:30,81,57.1,NaN,NaN,NaN,NaN,NaN,transportation_utilities,transportation_utilities,single_family,single_family,9404417997500635600,13234216672446100438,48.0,female,white_not_hispanic_or_latino,employed,employed_not_working,130236.0,private_auto,2,238767.0,three_plus,core,naics31_33,single_family,not_attending_school,bachelors_degree,owner,english,"4 (Tract 423.11, Orange, CA)","423.11 (Orange, CA)","Orange County, CA",California,"5 (Tract 626.10, Orange, CA)","626.10 (Orange, CA)","Orange County, CA",California,None


In [19]:
origins.origin_bgrp_2020.nunique()

591

In [20]:
unique_trips = origins.groupby('origin_bgrp_2020')['activity_id'].nunique()

In [21]:
unique_trips = unique_trips.reset_index(name='Unique_Count')

In [22]:
(unique_trips>>arrange(-_.Unique_Count)).head()

,origin_bgrp_2020,Unique_Count
255,"1 (Tract 9800.28, Los Angeles, CA)",56671
582,"5 (Tract 626.10, Orange, CA)",14241
63,"1 (Tract 2074, Los Angeles, CA)",13894
252,"1 (Tract 9800, Orange, CA)",11472
209,"1 (Tract 2679.01, Los Angeles, CA)",9511


In [23]:
# summary = replica_utils.return_score_summary(df_list)

In [24]:
# summary.columns = [col.replace('_', ' ').title() for col in summary.columns]


In [25]:
# summary.columns

In [26]:
# summary_melt =  pd.melt(
#         summary,
#         id_vars=["Trip Type"],
#         value_vars=['Trip Type', 'Total Trips',
#                     'N Auto Trips', 'Pct Auto Trips',
#                     'N Tranist Trips', 'Pct Transit Trips',
#                     'N Walking Trips', 'Pct Walking Trips',
#                     'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
#                     'Auto Mean Miles','Auto Median Miles', 'Auto Max Miles',
#                     'Transit Mean Minutes','Transit Median Minutes', 'Transit Max Minutes',
#                     'Transit Mean Miles','Transit Median Miles', 'Transit Max Miles',
#                     'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes',
#                     'Walking Mean Miles','Walking Median Miles', 'Walking Max Miles',
#                    ],
#         var_name="Metric",  # New column for original measurement names
#         # value_name="_Value"
# )

In [27]:
display(HTML("<h2><strong>Trip Type Summaries</strong></h2>"))

In [28]:
# from_cp_mode = replica_utils.get_mode_split(from_cp)
# to_cp_mode = replica_utils.get_mode_split(to_cp)

# modes_breakdown = pd.concat([to_cp_mode, from_cp_mode])

In [29]:
# modes_breakdown.columns = [col.replace('_', ' ').title() for col in modes_breakdown.columns]

In [30]:
# alt.Chart(modes_breakdown).mark_bar(size=150).encode(
#     x=alt.X("Trip Type:O", title = "Trip Type"),
#     y=alt.Y("Total Trips:Q", title="Total Number of Trips"),
#     color=alt.Color("Mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
#     tooltip = ['Total Trips', 'Mode']
#     ).properties(
#     width=600,  
#     height=300,
#     title='Modes by Trip Type')

In [31]:
# alt.Chart(modes_breakdown).mark_bar().encode(
#     x=alt.X('Trip Type:O', title ="Trip Type"),
#     y= alt.Y('Pct Trips:Q', title="Pct of Total Trips"),
#     color=alt.Color('Trip Type:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Type')),
#     column= alt.Column('Mode:N', title="Mode"),
#     tooltip=['Pct Trips', 'Mode', 'Trip Type']
# ).properties(width = 100, height = 400, title = "Modes by Trip Type").interactive()

In [32]:
display(HTML("<h3><strong>Closer Look at Auto, Transit & Walking Trips</strong></h3>"))

In [33]:
display(HTML("<strong>Tip:</strong> Use the dropdown menu to select different metrics to see on the chart."))

In [34]:
# metrics_list = summary_melt["Metric"].unique().tolist()

# metrics_dropdown = alt.binding_select(
#         options=metrics_list,
#         name="Metrics: ",
#     )
#     # Column that controls the bar charts
# xcol_param = alt.selection_point(
#         fields=["Metric"], value=metrics_list[0], bind=metrics_dropdown
#     )

# chart = (alt.Chart(summary_melt, title="Metric by Trip Types")
#         .mark_bar()
#         .encode(
#             x=alt.X("value"),
#             y=alt.Y("Trip Type"),
#             color=alt.Color("Trip Type",
#                                   scale=alt.Scale(
#                                       range=cp.CALITP_CATEGORY_BRIGHT_COLORS))
#         ).properties(width=400, height=350)
#     ).add_params(xcol_param).transform_filter(xcol_param)

# display(HTML("""
# <style>
# form.vega-bindings {
#   position: absolute;
#   left: 0px;
#   top: -0px;
# }
# </style>
# """))

# (chart)

In [35]:
# display(HTML("<br>"
#             "<br>"))

In [36]:
# summary_long_all_miles = pd.melt(
#     summary,
#     id_vars=["Trip Type"],
#     value_vars=[
#         'Auto Mean Miles', 'Auto Median Miles', 'Auto Max Miles', 
#         'Transit Mean Miles', 'Transit Median Miles', 'Transit Max Miles',
#         'Walking Mean Miles', 'Walking Median Miles', 'Walking Max Miles'],
#         var_name="Metric",  # New column for original measurement names
#         value_name="Miles")

In [37]:
# summary_long_all_min = pd.melt(
#     summary,
#     id_vars=["Trip Type"],
#     value_vars=[
#         'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
#         'Transit Mean Minutes', 'Transit Median Minutes', 'Transit Max Minutes',
#         'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes'],
#         var_name="Metric",  # New column for original measurement names
#         value_name="Mintues")

In [38]:
# display(HTML("<strong>Trip Length</strong>"))

In [39]:
# alt.Chart(summary_long_all_min).mark_bar().encode(
#     x="Mintues:Q",
#     y="Metric:O",
#     color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
#     row="Trip Type:O"
# ).properties(title="Travel Length by Trip Type & Mode")

In [40]:
# alt.Chart(summary_long_all_miles).mark_bar().encode(
#     x="Miles:Q",
#     y="Metric:O",
#     color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
#     row="Trip Type:O"
# ).properties(title="Travel Distance by Trip Type & Mode")

In [41]:
display(HTML("<h2><strong>Trip Timing</strong></h2>"))

In [42]:
display(HTML("<strong>Tip:</strong> You can zoom into each graph to better see the timing of the trips by mode"))

In [43]:
import datetime

In [44]:
# from_cp_time_melt, from_cp_time_melt_duration = replica_utils.return_time_metrics(from_cp, "trip_start_time", "trip_end_time")

In [45]:
# to_cp_time_melt, to_cp_time_melt_duration = replica_utils.return_time_metrics(to_cp, "trip_start_time", "trip_end_time")

In [46]:
# alt.Chart(from_cp_time_melt).mark_circle(size=150).encode(
#     x=alt.X('Time:T',axis=alt.Axis(format='%H:%M')),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Time'],
# ).properties(height=400, width=600, title="Trips Start and End Times for Trips From Cal Poly Humboldt").interactive()

In [47]:
# alt.Chart(to_cp_time_melt).mark_circle(size=150).encode(
#     x=alt.X('Time:T',axis=alt.Axis(format='%H:%M')),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Time'],
# ).properties(height=400, width=600, title="Trips Start and End Times for Trips To Cal Poly Humboldt").interactive()

In [48]:
# alt.Chart(from_cp_time_melt_duration).mark_circle(size=150).encode(
#     x=alt.X('Value:Q', title="Minutes"),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Value']
# ).properties(height=400, width=600, title="Trip Duration for Trips From Cal Poly Humboldt").interactive()

In [49]:
# alt.Chart(to_cp_time_melt_duration).mark_circle(size=150).encode(
#     x=alt.X('Value:Q', title="Minutes"),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Value']
# ).properties(height=400, width=600, title="Trip Duration for Trips To Cal Poly Humboldt").interactive()

In [50]:
# n_trips_to_cp = (to_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry"]).agg(
#         {"activity_id": "nunique"}).reset_index()
# n_trips_to_cp = n_trips_to_cp.set_geometry("geometry")


In [51]:
# n_trips_from_cp = (from_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry"]).agg(
#         {"activity_id": "nunique"}).reset_index()
# n_trips_from_cp = n_trips_from_cp.set_geometry("geometry")


In [52]:
# n_trips_to_cp['trip_type'] = 'Traveling To Cal Poly'
# n_trips_from_cp['trip_type'] = 'Traveling From Cal Poly'

In [53]:
# ntrips = pd.concat([n_trips_from_cp, n_trips_to_cp])

In [54]:
display(HTML("<h2><strong>Trips by Census Block Groups</strong></h2>"))

In [55]:
# with get_fs().open(f"gs://calitp-analytics-data/data-analyses/traffic_ops/ca_transit_routes.parquet") as f:
#         routes = to_snakecase(gpd.read_parquet(f))

In [56]:
# routes = routes[routes["agency"]=="Humboldt Transit Authority"]

In [57]:
# routes.sample()

In [58]:
# routes.columns = [col.replace('_', ' ').title() for col in routes.columns]


In [59]:
# routes = routes.rename(columns={"Geometry":"geometry", "Shn Route":"SHN Route"})

In [60]:
# minx, miny, maxx, maxy = routes.total_bounds
# sw = [miny, minx]
# ne = [maxy, maxx]

In [61]:
# n_trips_from_cp = n_trips_from_cp.rename(columns={"destination_bgrp_2020":"Census BlockGroup","activity_id":"Number of Trips", "trip_type":"Trip Type"})
# n_trips_to_cp = n_trips_to_cp.rename(columns={"origin_bgrp_2020":"Census BlockGroup", "activity_id":"Number of Trips", "trip_type":"Trip Type"})

In [62]:
display(HTML("<h4>Trips From Cal Poly</h4>"))

In [63]:
# m = n_trips_from_cp.explore(name="Trips from Cal Poly", column="Number of Trips", 
#                 scheme="NaturalBreaks",
#                  k=10,
#                 cmap="YlOrRd")
# m = routes.explore(m=m, column="Route Name", name="HTA Routes", tooltip=["Agency","Route Name", "SHN Route"], cmap="tab20" )
# # this is completely optional
# folium.LayerControl().add_to(m)

# m.fit_bounds([sw, ne])
    
# display(m)

In [64]:
display(HTML("<h3>Trips To Cal Poly</h3>"))

In [65]:
# m = n_trips_to_cp.explore(name="Trips To Cal Poly", column="Number of Trips", 
#                 scheme="NaturalBreaks",
#                  k=10,
#                 cmap="YlOrRd")
# m = routes.explore(m=m, column="Route Name", name="HTA Routes", tooltip=["Agency","Route Name", "SHN Route"])
# # this is completely optional
# folium.LayerControl().add_to(m)

# m

In [66]:
# n_trips_from_cp_mode = (
#     from_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry", "primary_mode"]).agg(
#     n_trips=("activity_id", "nunique"),
    
#     trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#     trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
    
#     trip_distance_miles_median=('trip_distance_miles', 'median'),
#     trip_distance_miles_mean=('trip_distance_miles', 'mean'),
    
# #     trip_start_time_mean=('trip_start_time', 'mean'),
# #     trip_start_time_median=('trip_start_time', 'median'),
    
# #     trip_end_time_mean=('trip_end_time', 'mean'),
# #     trip_end_time_median=('trip_end_time', 'median'),

#         ).reset_index()
    
# n_trips_from_cp_mode = n_trips_from_cp_mode.set_geometry("geometry")

In [67]:
# n_trips_to_cp_mode = (
#     to_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry", "primary_mode"]).agg(
#     n_trips=("activity_id", "nunique"),
    
#     trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#     trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
    
#     trip_distance_miles_median=('trip_distance_miles', 'median'),
#     trip_distance_miles_mean=('trip_distance_miles', 'mean'),
    
# #     trip_start_time_mean=('trip_start_time', 'mean'),
# #     trip_start_time_median=('trip_start_time', 'median'),
    
# #     trip_end_time_mean=('trip_end_time', 'mean'),
# #     trip_end_time_median=('trip_end_time', 'median'),

#         ).reset_index()
    
# n_trips_to_cp_mode = n_trips_to_cp_mode.set_geometry("geometry")

In [68]:
# n_trips_from_cp_mode.head()

In [69]:
# alt.Chart(n_trips_from_cp_mode).mark_bar().encode(
#     x=alt.X('primary_mode', title ="Trip Mode"),
#     y= alt.Y('n_trips', title="Number of Trips"),
#     # color=alt.Color('destination_bgrp_2020:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Type')),
# ).properties(width = 500, height = 400, title = "Modes by Trip Type")

In [70]:
# n_trips_from_cp_mode.sample()

In [71]:
# unique_modes = from_cp.primary_mode.unique()

In [72]:
# unique_modes

In [73]:
# unique_modes = ['private_auto', 'public_transit', 'walking']

In [74]:
display(HTML("<h2>Trips Traveling To Cal Poly</h2>"))

In [75]:
# ##hashing out for now for saving
# replica_utils.return_mode_map(n_trips_to_cp_mode, routes, unique_modes, "to")

In [76]:
display(HTML("<h4>Trips Traveling From Cal Poly</h4>"))

In [78]:
###hashing out for now for saving
# replica_utils.return_mode_map(n_trips_from_cp_mode, routes, unique_modes, "from")

In [79]:
display(HTML("<h2>Raw Trip Data Sample</h2>"))

In [80]:
origins.sample(3)

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,geometry
21230,1613820683350241208,"1 (Tract 9800.28, Los Angeles, CA)","9800.28 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 17.08, Orange, CA)","17.08 (Orange, CA)","Orange County, CA",California,private_auto,work,other_activity_type,15:51:00,17:07:17,76,30.3,unknown_vehicle_type,other_non_bev,NaN,NaN,NaN,transportation_utilities,transportation_utilities,mixed_use,non_retail_attraction,10688899277021652359,14157002805656581637,56.0,female,white_not_hispanic_or_latino,employed,in_person,32457.0,private_auto,4,481293.0,three_plus,core,naics61,single_family,not_attending_school,some_college,owner,english,"1 (Tract 218.23, Orange, CA)","218.23 (Orange, CA)","Orange County, CA",California,"2 (Tract 17.08, Orange, CA)","17.08 (Orange, CA)","Orange County, CA",California,None
223271,2789322342064316225,"5 (Tract 5762, Los Angeles, CA)","5762 (Los Angeles, CA)","Los Angeles County, CA",California,"3 (Tract 406.16, Riverside, CA)","406.16 (Riverside, CA)","Riverside County, CA",California,private_auto,home,shop,17:15:00,18:22:28,67,43.0,NaN,NaN,NaN,NaN,NaN,office,office,single_family,single_family,18051153869745590104,3675490095616116134,63.0,female,white_not_hispanic_or_latino,employed,in_person,68817.0,private_auto,6,202782.0,three_plus,core,naics92,single_family,not_attending_school,some_college,owner,english,"3 (Tract 406.16, Riverside, CA)","406.16 (Riverside, CA)","Riverside County, CA",California,"2 (Tract 5760.01, Los Angeles, CA)","5760.01 (Los Angeles, CA)","Los Angeles County, CA",California,None
124727,6565722855917529400,"1 (Tract 2260.02, Los Angeles, CA)","2260.02 (Los Angeles, CA)","Los Angeles County, CA",California,"3 (Tract 1411.01, Los Angeles, CA)","1411.01 (Los Angeles, CA)","Los Angeles County, CA",California,public_transit,shop,work,16:18:00,17:16:59,58,15.0,unknown_vehicle_type,unknown_fuel_type,"bus, subway, bus","LADOTDT, Metro - Los Angeles, Metro - Los Angeles","DASH E, Metro B Line, Metro Local Line",mixed_use,industrial,retail,retail,8109946778975582500,18356643618583571175,64.0,male,black_not_hispanic_or_latino,employed,in_person,9420.0,public_transit,1,9420.0,one,core,naics4481,single_family,not_attending_school,high_school,owner,english,"2 (Tract 1245, Los Angeles, CA)","1245 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 2260.02, Los Angeles, CA)","2260.02 (Los Angeles, CA)","Los Angeles County, CA",California,None
